# 🧟 Zombie Survival AI Challenge — Fixed Version

You are building a tiny neural-network brain for a zombie apocalypse survivor.

Every day, the AI receives information about the situation and chooses one action:

- 🔫 Shoot
- 🏃 Run
- 🩹 Heal
- 🔍 Search

Your job is to tune the neural-network **weights** so your survivor lasts as long as possible.



```text
low_health = 1 means very injured
low_health = 0 means healthy
```

So if you want the AI to heal when health is low, you give `low_health` a **positive weight** for the `Heal` action.


## Inputs and outputs

The AI sees 7 inputs:

| Input | Meaning |
|---|---|
| `zombie_closeness` | 0 = far away, 1 = right next to you |
| `horde_size` | 0 = few zombies, 1 = huge horde |
| `low_health` | 0 = healthy, 1 = almost dead |
| `low_ammo` | 0 = lots of ammo, 1 = no ammo |
| `low_food` | 0 = lots of food, 1 = starving |
| `medkits_available` | 0 = no medkits, 1 = many medkits |
| `rescue_soon` | 0 = rescue far away, 1 = rescue soon |

The outputs are:

```text
[Shoot, Run, Heal, Search]
```

The AI computes:

```python
scores = inputs @ weights
```

Then it chooses the action with the highest score.


In [1]:
import numpy as np
import random
import time

INPUT_NAMES = [
    "zombie_closeness",
    "horde_size",
    "low_health",
    "low_ammo",
    "low_food",
    "medkits_available",
    "rescue_soon",
]

ACTION_NAMES = ["Shoot", "Run", "Heal", "Search"]
ACTION_EMOJIS = {
    "Shoot": "🔫",
    "Run": "🏃",
    "Heal": "🩹",
    "Search": "🔍",
}

def clamp(x, low, high):
    return max(low, min(high, x))

def bar(value, width=10):
    filled = int(round(clamp(value, 0, 1) * width))
    return "█" * filled + "░" * (width - filled)

print("Zombie Survival AI loaded.")


Zombie Survival AI loaded.


# Part 1 — Design your AI

Edit the weight matrix below.

Rows = inputs.  
Columns = actions.

```text
[Shoot, Run, Heal, Search]
```

Positive weight = this input encourages that action.  
Negative weight = this input discourages that action.

Examples:

```python
# If zombie_closeness is high, encourage Shoot and Run
[5, 4, -3, -4]

# If low_ammo is high, discourage Shoot and encourage Search
[-8, 1, 0, 6]
```


In [8]:
# =====================================================
# EDIT THIS AI
# =====================================================
# Columns:             Shoot   Run   Heal  Search

weights = np.array([
    [ 8.0,   4.0,  -3.0,  -4.0],   # zombie_closeness
    [ 4.0,   6.0,  -3.0,  -5.0],   # horde_size
    [-2.0,   3.0,   8.0,  -2.0],   # low_health
    [-8.0,   1.0,   0.0,   6.0],   # low_ammo
    [-1.0,   1.0,   0.0,   7.0],   # low_food
    [ 0.0,  -1.0,   7.0,  -2.0],   # medkits_available
    [-1.0,   5.0,   1.0,  -3.0],   # rescue_soon
])

def choose_action(inputs, weights, randomness=0.05, verbose=False):
    inputs = np.array(inputs)
    scores = inputs @ weights

    # Small randomness means the AI is not always perfectly predictable.
    noisy_scores = scores + np.random.normal(0, randomness, size=len(scores))

    action_index = int(np.argmax(noisy_scores))
    action = ACTION_NAMES[action_index]

    if verbose:
        print("Inputs:")
        for name, value in zip(INPUT_NAMES, inputs):
            print(f"{name:20s}: {value:.2f} {bar(value)}")

        print("\nAction scores:")
        for name, score, noisy_score in zip(ACTION_NAMES, scores, noisy_scores):
            print(f"{ACTION_EMOJIS[name]} {name:7s}: {score:6.2f}  noisy={noisy_score:6.2f}")

        print(f"\nAI decision: {ACTION_EMOJIS[action]} {action}")

    return action, scores


# Part 2 — Try one situation

Before running a campaign, test the AI on individual situations.

Important: `low_health = 0.9` means the survivor is badly injured.  
So if the AI is sensible, that should strongly encourage `Heal`, especially when medkits are available.


In [6]:
situation = [
    0.3,  # zombie_closeness
    0.2,  # horde_size
    0.9,  # low_health
    0.4,  # low_ammo
    0.3,  # low_food
    1.0,  # medkits_available
    0.2,  # rescue_soon
]

choose_action(situation, weights, randomness=0.0, verbose=True)


Inputs:
zombie_closeness    : 0.30 ███░░░░░░░
horde_size          : 0.20 ██░░░░░░░░
low_health          : 0.90 █████████░
low_ammo            : 0.40 ████░░░░░░
low_food            : 0.30 ███░░░░░░░
medkits_available   : 1.00 ██████████
rescue_soon         : 0.20 ██░░░░░░░░

Action scores:
🔫 Shoot  :  -3.20  noisy= -3.20
🏃 Run    :   5.80  noisy=  5.80
🩹 Heal   :  12.90  noisy= 12.90
🔍 Search :  -2.10  noisy= -2.10

AI decision: 🩹 Heal


('Heal', array([-3.2,  5.8, 12.9, -2.1]))

## Mini test situations

Try to predict what your AI will do before you run the cell.


In [9]:
mini_tests = {
    "Close zombie, lots of ammo":      [0.9, 0.2, 0.1, 0.0, 0.3, 0.2, 0.1],
    "Huge horde, low health":          [0.8, 1.0, 0.8, 0.4, 0.4, 0.2, 0.2],
    "Quiet day, low supplies":         [0.1, 0.0, 0.2, 0.9, 0.9, 0.0, 0.3],
    "Injured with medkit":             [0.3, 0.2, 0.9, 0.5, 0.4, 1.0, 0.4],
    "Rescue soon, zombies nearby":     [0.7, 0.6, 0.5, 0.6, 0.5, 0.3, 1.0],
    "No ammo but zombies are close":   [0.9, 0.6, 0.3, 1.0, 0.4, 0.1, 0.2],
}

for name, test in mini_tests.items():
    action, scores = choose_action(test, weights, randomness=0.0)
    print(f"{name:32s} -> {ACTION_EMOJIS[action]} {action}")


Close zombie, lots of ammo       -> 🔫 Shoot
Huge horde, low health           -> 🏃 Run
Quiet day, low supplies          -> 🔍 Search
Injured with medkit              -> 🩹 Heal
Rescue soon, zombies nearby      -> 🏃 Run
No ammo but zombies are close    -> 🏃 Run


# Part 3 — The campaign simulator

The survivor tries to survive until rescue arrives.

Each day:

1. A random situation appears.
2. Your neural network chooses an action.
3. The action changes health, ammo, food, medkits, and score.
4. If health reaches 0, the survivor dies.

The world is random, so the same AI can have different stories.


In [10]:
def make_world_state(day, total_days):
    # The world gets more dangerous as days pass.
    danger = day / total_days

    zombie_closeness = clamp(random.random() * 0.8 + danger * 0.25, 0, 1)
    horde_size = clamp(random.random() * 0.7 + danger * 0.35, 0, 1)
    rescue_soon = day / total_days

    return zombie_closeness, horde_size, rescue_soon

def make_inputs(player, day, total_days):
    zombie_closeness, horde_size, rescue_soon = make_world_state(day, total_days)

    # FIXED: these inputs now mean "danger/need" rather than "resource amount".
    low_health = 1 - player["health"] / 100
    low_ammo = 1 - player["ammo"] / 12
    low_food = 1 - player["food"] / 8
    medkits_available = player["medkits"] / 4

    inputs = [
        zombie_closeness,
        horde_size,
        low_health,
        low_ammo,
        low_food,
        medkits_available,
        rescue_soon,
    ]

    inputs = [clamp(x, 0, 1) for x in inputs]
    return inputs

def starting_player():
    return {
        "health": 100.0,
        "ammo": 6,
        "food": 4,
        "medkits": 1,
        "rescued_people": 0,
        "score": 0.0,
        "alive": True,
        "bullets_fired": 0,
        "days_survived": 0,
    }


In [11]:
def resolve_action(player, action, inputs, day, total_days):
    zombie_closeness, horde_size, low_health, low_ammo, low_food, medkits_available, rescue_soon = inputs

    event_lines = []

    # Daily food cost
    player["food"] -= 1
    if player["food"] < 0:
        player["health"] -= 8
        player["food"] = 0
        event_lines.append("You had no food. Hunger weakened you.")

    # Convert zombie inputs into danger.
    danger = 0.55 * zombie_closeness + 0.45 * horde_size

    if action == "Shoot":
        event_lines.append("You raise your weapon and fire.")

        if player["ammo"] <= 0:
            player["health"] -= 18 + 20 * danger
            player["score"] -= 20
            event_lines.append("Click. No ammo. The zombies close in.")
        else:
            player["ammo"] -= 1
            player["bullets_fired"] += 1

            # More ammo usually means calmer/better shooting.
            ammo_amount = 1 - low_ammo
            hit_chance = clamp(0.75 - 0.35 * horde_size + 0.15 * ammo_amount, 0.15, 0.9)

            if random.random() < hit_chance:
                player["score"] += 18 + 10 * danger
                event_lines.append("Hit! The nearest zombie drops.")

                if random.random() < 0.12 + 0.18 * danger:
                    player["rescued_people"] += 1
                    player["score"] += 25
                    event_lines.append("You rescued a trapped survivor!")
            else:
                damage = 8 + 22 * danger
                player["health"] -= damage
                player["score"] -= 8
                event_lines.append("Miss! The shot echoes through the street.")

    elif action == "Run":
        event_lines.append("You sprint away through the ruined streets.")

        health_amount = 1 - low_health
        escape_chance = clamp(0.80 - 0.35 * zombie_closeness - 0.20 * horde_size + 0.10 * health_amount, 0.15, 0.95)

        if random.random() < escape_chance:
            player["score"] += 8
            event_lines.append("You escape.")

            if random.random() < 0.25:
                lost = random.choice(["food", "ammo"])
                if player[lost] > 0:
                    player[lost] -= 1
                    event_lines.append(f"While running, you dropped 1 {lost}.")
        else:
            damage = 12 + 25 * danger
            player["health"] -= damage
            player["score"] -= 12
            event_lines.append("You stumble. The horde catches up.")

    elif action == "Heal":
        event_lines.append("You stop to treat your wounds.")

        if player["medkits"] <= 0:
            player["score"] -= 8
            player["health"] -= 7 + 15 * danger
            event_lines.append("No medkits left. You wasted precious time.")
        else:
            player["medkits"] -= 1
            heal_amount = random.randint(22, 38)
            player["health"] = min(100, player["health"] + heal_amount)
            player["score"] += 10
            event_lines.append(f"You used a medkit and recovered {heal_amount} health.")

            if danger > 0.55 and random.random() < danger:
                damage = 8 + 18 * danger
                player["health"] -= damage
                event_lines.append("But healing took too long. Zombies attacked.")

    elif action == "Search":
        event_lines.append("You search nearby buildings for supplies.")

        search_success = clamp(0.65 - 0.35 * danger, 0.10, 0.85)

        if random.random() < search_success:
            found = random.choice(["ammo", "food", "medkits", "survivor"])

            if found == "ammo":
                amount = random.randint(2, 5)
                player["ammo"] += amount
                player["score"] += 10
                event_lines.append(f"You found {amount} bullets.")

            elif found == "food":
                amount = random.randint(1, 3)
                player["food"] += amount
                player["score"] += 8
                event_lines.append(f"You found {amount} food.")

            elif found == "medkits":
                player["medkits"] += 1
                player["score"] += 12
                event_lines.append("You found a medkit.")

            elif found == "survivor":
                player["rescued_people"] += 1
                player["score"] += 30
                event_lines.append("You found and rescued a survivor hiding nearby!")
        else:
            player["score"] -= 5
            event_lines.append("You found nothing.")

            if random.random() < danger:
                damage = 8 + 20 * danger
                player["health"] -= damage
                event_lines.append("The noise attracted zombies.")

    # Clamp resources.
    player["health"] = clamp(player["health"], 0, 100)
    player["ammo"] = int(clamp(player["ammo"], 0, 20))
    player["food"] = int(clamp(player["food"], 0, 12))
    player["medkits"] = int(clamp(player["medkits"], 0, 6))

    if player["health"] <= 0:
        player["alive"] = False
        event_lines.append("The zombies overwhelmed you.")

    return event_lines


In [12]:
def describe_day(day, total_days, inputs, action, event_lines, player):
    zombie_closeness, horde_size, low_health, low_ammo, low_food, medkits_available, rescue_soon = inputs

    lines = []
    lines.append(f"🧟 DAY {day}/{total_days}")

    if horde_size > 0.75:
        lines.append("A huge horde blocks the road.")
    elif zombie_closeness > 0.75:
        lines.append("A zombie is almost on top of you.")
    elif zombie_closeness < 0.25 and horde_size < 0.25:
        lines.append("The streets are strangely quiet.")
    else:
        lines.append("You hear movement nearby.")

    lines.append(f"❤️ Health: {player['health']:5.1f} | 🔫 Ammo: {player['ammo']} | 🍖 Food: {player['food']} | 🩹 Medkits: {player['medkits']}")
    lines.append(f"AI decision: {ACTION_EMOJIS[action]} {action}")

    for line in event_lines:
        lines.append("  " + line)

    return "\n".join(lines)

def run_campaign(weights, seed=1, total_days=18, narrate=False, pause=0.0, important_only=True):
    random.seed(seed)
    np.random.seed(seed)

    player = starting_player()

    for day in range(1, total_days + 1):
        if not player["alive"]:
            break

        inputs = make_inputs(player, day, total_days)
        action, scores = choose_action(inputs, weights, randomness=0.05)
        event_lines = resolve_action(player, action, inputs, day, total_days)

        player["days_survived"] = day
        player["score"] += 5

        important = (
            day == 1 or
            day == total_days or
            player["health"] < 35 or
            inputs[0] > 0.75 or
            inputs[1] > 0.75 or
            "rescued" in " ".join(event_lines).lower() or
            not player["alive"]
        )

        if narrate and (not important_only or important):
            print(describe_day(day, total_days, inputs, action, event_lines, player))
            print("-" * 60)
            if pause > 0:
                time.sleep(pause)

    if player["alive"] and player["days_survived"] >= total_days:
        player["score"] += 100
        final_status = "Rescued"
    elif player["alive"]:
        final_status = "Alive"
    else:
        final_status = "Dead"

    player["score"] += 25 * player["rescued_people"]
    player["score"] += player["health"] * 0.5
    player["score"] -= player["bullets_fired"] * 1.5

    summary = {
        "status": final_status,
        "score": round(player["score"], 1),
        "days_survived": player["days_survived"],
        "health": round(player["health"], 1),
        "ammo": player["ammo"],
        "food": player["food"],
        "medkits": player["medkits"],
        "rescued_people": player["rescued_people"],
        "bullets_fired": player["bullets_fired"],
    }

    return summary


# Part 4 — Run one narrated campaign


In [13]:
summary = run_campaign(weights, seed=7, total_days=18, narrate=True, pause=0.0, important_only=True)

print("FINAL SUMMARY")
for key, value in summary.items():
    print(f"{key:16s}: {value}")


🧟 DAY 1/18
You hear movement nearby.
❤️ Health:  87.9 | 🔫 Ammo: 6 | 🍖 Food: 3 | 🩹 Medkits: 1
AI decision: 🔍 Search
  You search nearby buildings for supplies.
  You found nothing.
  The noise attracted zombies.
------------------------------------------------------------
🧟 DAY 9/18
You hear movement nearby.
❤️ Health:  29.1 | 🔫 Ammo: 9 | 🍖 Food: 0 | 🩹 Medkits: 1
AI decision: 🏃 Run
  You had no food. Hunger weakened you.
  You sprint away through the ruined streets.
  You escape.
  While running, you dropped 1 ammo.
------------------------------------------------------------
🧟 DAY 10/18
A zombie is almost on top of you.
❤️ Health:   0.0 | 🔫 Ammo: 9 | 🍖 Food: 0 | 🩹 Medkits: 1
AI decision: 🏃 Run
  You had no food. Hunger weakened you.
  You sprint away through the ruined streets.
  You stumble. The horde catches up.
  The zombies overwhelmed you.
------------------------------------------------------------
FINAL SUMMARY
status          : Dead
score           : 79.0
days_survived   : 10
h

# Part 5 — Evaluate your AI

One campaign can be lucky or unlucky.

So we evaluate across many campaigns and use the average.


In [14]:
def evaluate_ai(weights, seeds=range(20), total_days=18):
    results = []

    for seed in seeds:
        summary = run_campaign(weights, seed=seed, total_days=total_days, narrate=False)
        results.append(summary)

    avg_score = sum(r["score"] for r in results) / len(results)
    rescue_rate = sum(1 for r in results if r["status"] == "Rescued") / len(results)
    avg_days = sum(r["days_survived"] for r in results) / len(results)
    avg_rescued = sum(r["rescued_people"] for r in results) / len(results)

    print(f"Average score:       {avg_score:.1f}")
    print(f"Rescue success rate: {rescue_rate*100:.1f}%")
    print(f"Average days alive:  {avg_days:.1f}")
    print(f"Average rescued:     {avg_rescued:.1f}")

    return {
        "avg_score": avg_score,
        "rescue_rate": rescue_rate,
        "avg_days": avg_days,
        "avg_rescued": avg_rescued,
        "results": results,
    }

evaluation = evaluate_ai(weights, seeds=range(30), total_days=18)


Average score:       42.8
Rescue success rate: 0.0%
Average days alive:  8.8
Average rescued:     0.0


# Part 6 — Strategy ideas

Try to create a personality.

## The Soldier

Shoots often. Good when ammo is high, risky when ammo is low.

## The Runner

Runs from large hordes. Usually survives, but may rescue fewer people.

## The Medic

Heals when injured and medkits are available.

## The Loot Goblin

Searches when supplies are low. Can become strong, but may die searching.

## The Hero

Takes risks to rescue people. Can score highly, but may die early.


# Part 7 — Group submission

Each group should paste their final weights into the dictionary below.

Give your AI a team name.


In [33]:
group_ais = {
    "Example AI": weights,

    "The Soldier": np.array([
        [ 6.0,   1.0,  -4.0,  -4.0],
        [ 5.0,   0.0,  -3.0,  -5.0],
        [-2.0,   0.0,   5.0,  -1.0],
        [-9.0,   0.0,   0.0,   5.0],
        [-1.0,   0.0,   0.0,   5.0],
        [ 0.0,  -1.0,   6.0,  -2.0],
        [-1.0,   4.0,   1.0,  -3.0],
    ]),

    "Loot Goblin": np.array([
        [ 8.0,   2.0,  -3.0,  -5.0],
        [ 1.0,   7.0,  -3.0,  -6.0],
        [-2.0,   3.0,   0.0,  -1.0],
        [-8.0,   1.0,   0.0,   8.0],
        [-2.0,   1.0,   0.0,   9.0],
        [ 0.0,  -1.0,   0.0,  -3.0],
        [-2.0,   5.0,   0.0,  -4.0],
    ]),

    "Medic Runner": np.array([
        [ 0.0,   6.0,  -4.0,  -5.0],
        [ 1.0,   8.0,  -4.0,  -6.0],
        [-3.0,   4.0,  10.0,  -2.0],
        [-7.0,   2.0,   0.0,   5.0],
        [-1.0,   2.0,   0.0,   6.0],
        [ 0.0,  -1.0,   9.0,  -3.0],
        [-2.0,   7.0,   2.0,  -5.0],
    ]),
}


# Part 8 — Leaderboard

This compares the average performance of each AI over the same campaigns.


In [19]:
def leaderboard(group_ais, seeds=range(50), total_days=18):
    table = []

    for name, ai_weights in group_ais.items():
        scores = []
        rescued_runs = 0
        rescued_people = 0
        days = 0

        for seed in seeds:
            summary = run_campaign(ai_weights, seed=seed, total_days=total_days, narrate=False)
            scores.append(summary["score"])
            if summary["status"] == "Rescued":
                rescued_runs += 1
            rescued_people += summary["rescued_people"]
            days += summary["days_survived"]

        table.append({
            "name": name,
            "avg_score": sum(scores) / len(scores),
            "best_score": max(scores),
            "rescue_rate": rescued_runs / len(seeds),
            "avg_rescued_people": rescued_people / len(seeds),
            "avg_days": days / len(seeds),
        })

    table = sorted(table, key=lambda x: x["avg_score"], reverse=True)

    print("🏆 ZOMBIE SURVIVAL AI LEADERBOARD 🏆")
    print()
    for place, row in enumerate(table, start=1):
        print(
            f"{place}. {row['name']:15s} | "
            f"avg score {row['avg_score']:7.1f} | "
            f"best {row['best_score']:7.1f} | "
            f"rescued {row['rescue_rate']*100:5.1f}% | "
            f"avg people {row['avg_rescued_people']:4.1f}"
        )

    return table

results_table = leaderboard(group_ais, seeds=range(50), total_days=18)


🏆 ZOMBIE SURVIVAL AI LEADERBOARD 🏆

1. The Soldier     | avg score   115.8 | best   288.2 | rescued   4.0% | avg people  0.7
2. Example AI      | avg score    50.8 | best   189.0 | rescued   0.0% | avg people  0.1
3. Loot Goblin     | avg score    48.5 | best   189.0 | rescued   0.0% | avg people  0.1
4. Medic Runner    | avg score    43.7 | best   142.0 | rescued   0.0% | avg people  0.0


# Part 9 — Grand Final Story Mode

Every team faces the **same world** using the same random seed.

The situations are identical, but the AI decisions are different.


In [32]:
def grand_final_story(group_ais, seed=999, total_days=18, pause=1.0):
    print("🎬 GRAND FINAL: SAME APOCALYPSE, DIFFERENT AI STRATEGIES")
    print("=" * 70)
    print()

    final_summaries = {}

    for name, ai_weights in group_ais.items():
        print(f"TEAM: {name}")
        print("=" * 70)

        summary = run_campaign(
            ai_weights,
            seed=seed,
            total_days=total_days,
            narrate=True,
            pause=pause,
            important_only=True
        )

        final_summaries[name] = summary

        print("FINAL SUMMARY")
        for key, value in summary.items():
            print(f"{key:16s}: {value}")
        print("\n" + "#" * 70 + "\n")

    print("🏆 GRAND FINAL RESULTS 🏆")
    ranking = sorted(final_summaries.items(), key=lambda x: x[1]["score"], reverse=True)

    for place, (name, summary) in enumerate(ranking, start=1):
        print(
            f"{place}. {name:15s} | "
            f"score {summary['score']:7.1f} | "
            f"status {summary['status']:8s} | "
            f"days {summary['days_survived']:2d} | "
            f"rescued people {summary['rescued_people']}"
        )

    return final_summaries

grand_final = grand_final_story(group_ais, seed=999, total_days=18, pause=0.0)


🎬 GRAND FINAL: SAME APOCALYPSE, DIFFERENT AI STRATEGIES

TEAM: Example AI
🧟 DAY 1/18
You hear movement nearby.
❤️ Health:  78.4 | 🔫 Ammo: 6 | 🍖 Food: 3 | 🩹 Medkits: 1
AI decision: 🏃 Run
  You sprint away through the ruined streets.
  You stumble. The horde catches up.
------------------------------------------------------------
🧟 DAY 4/18
A huge horde blocks the road.
❤️ Health:  55.4 | 🔫 Ammo: 6 | 🍖 Food: 0 | 🩹 Medkits: 1
AI decision: 🏃 Run
  You sprint away through the ruined streets.
  You escape.
------------------------------------------------------------
🧟 DAY 6/18
A zombie is almost on top of you.
❤️ Health:  12.3 | 🔫 Ammo: 6 | 🍖 Food: 0 | 🩹 Medkits: 1
AI decision: 🏃 Run
  You had no food. Hunger weakened you.
  You sprint away through the ruined streets.
  You stumble. The horde catches up.
------------------------------------------------------------
🧟 DAY 7/18
The streets are strangely quiet.
❤️ Health:  32.3 | 🔫 Ammo: 6 | 🍖 Food: 0 | 🩹 Medkits: 0
AI decision: 🩹 Heal
  You had

# Part 10 — Reflection

Discuss:

1. Which strategy survived best?
2. Which strategy rescued the most people?
3. Did the highest average score also win story mode?
4. Which weights mattered most?
5. Did any AI make a decision that looked silly?
6. How is this similar to real machine learning?

Key idea:

> In this activity, you manually tuned the neural-network weights.
>
> In real machine learning, the computer tunes the weights automatically using data.
